# Pengklasifikasi teks alternatif: gzip-kNN, BM25-kNN, Rocchio, Complement NB, PA, kernel RFF
**Kapan dipakai:** model utama (TF-IDF + linear / transformer) sudah ada dan ensemble butuh
anggota yang **salahnya beda**. Model di sini jarang menang sendirian, tapi sering menambah skor
saat digabung (`ensemble_oof`, `bobot_ensemble_metaheuristik`).

| model | ide | sumber |
|---|---|---|
| gzip-kNN | jarak kompresi NCD: teks mirip = gabungannya ringkas | Jiang dkk. 2023 |
| BM25-kNN | tetangga terdekat dengan skor mesin pencari BM25 | Robertson & Zaragoza 2009 |
| Rocchio | kemiripan kosinus ke pusat (rata-rata) tiap kelas | klasik temu-kembali |
| Complement NB | NB dari kelas komplemen, kuat untuk kelas timpang | Rennie dkk. 2003 |
| passive-aggressive | update hanya saat salah, langkah sebesar kesalahan | Crammer dkk. 2006 |
| kernel RFF | fitur Fourier acak ≈ SVM kernel RBF, tetap linear & cepat | Rahimi & Recht 2007 |

Catatan jujur gzip-kNN: angka makalah asli sempat dikoreksi (salah hitung top-k), jadi harapkan
skor di bawah TF-IDF. Kekuatannya: tahan typo/singkatan dan tanpa kosakata.

In [ ]:
import os
import time
import zlib

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import normalize

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/pengklasifikasi_alternatif"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/classification"    # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/emot_train.csv"                  # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/emot_test.csv"                    # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/emot_test_labels.csv"              # GANTI saat lomba (kunci, kalau ada)
ID_COL = "id"                                # GANTI saat lomba
TARGET = "label"                             # GANTI saat lomba
TEXT_COLS = ["text"]                         # GANTI saat lomba
NCD_POOL = 500               # baris pembanding gzip per lipatan (±20 µs per pasang, ±1 menit)
NCD_K = 15
BM25_K = 25
ROCCHIO_TEMP = 20.0          # makin besar = peluang makin tajam
RFF_GAMMA = 1.0
N_FOLDS = 5
SEED = 42


def metric(truth, pred):
    return f1_score(truth, pred.argmax(axis=1), average="macro")      # GANTI: metrik lomba


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])))

## Baca data + matriks bersama (train + test, tanpa label)

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
texts = []
for table in [train, test]:
    text = table[TEXT_COLS].fillna("")
    texts.append(text.astype(str).agg(" . ".join, axis=1))
all_text = pd.concat(texts, ignore_index=True)
CLASSES, y = np.unique(train[TARGET].astype(str), return_inverse=True)
n_train = len(train)
n_test = len(test)
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train, y))
counts = CountVectorizer(ngram_range=(1, 2), min_df=2).fit_transform(all_text)
word = TfidfTransformer(sublinear_tf=True).fit_transform(counts)
char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True,
                       max_features=100000).fit_transform(all_text)
tfidf = hstack([word, char]).tocsr()
svd = normalize(TruncatedSVD(128, random_state=SEED).fit_transform(tfidf))
print("baris:", len(all_text), "| kelas:", list(CLASSES), "| tfidf:", tfidf.shape)

## Alat bersama
Tiap model = fungsi `(idx_fit, idx_query) → peluang`; indeks menunjuk baris `all_text`
(train dulu, lalu test). OOF memakai lipatan sama untuk semua model.

In [ ]:
def softmax(scores):
    shifted = np.exp(scores - scores.max(axis=1, keepdims=True))
    return shifted / shifted.sum(axis=1, keepdims=True)


def knn_vote(similarity, pool_labels, k):
    top = np.argpartition(-similarity, k, axis=1)[:, :k]
    proba = np.full((len(similarity), len(CLASSES)), 1e-6)
    rows = np.arange(len(similarity))
    for column in range(k):
        weight = np.clip(similarity[rows, top[:, column]], 0, None)
        proba[rows, pool_labels[top[:, column]]] += weight
    return proba / proba.sum(axis=1, keepdims=True)


def run_member(predict):
    oof = np.zeros((n_train, len(CLASSES)))
    test_pred = np.zeros((n_test, len(CLASSES)))
    test_rows = np.arange(n_train, n_train + n_test)
    for idx_fit, idx_val in folds:
        oof[idx_val] = predict(idx_fit, idx_val)
        test_pred += predict(idx_fit, test_rows) / N_FOLDS
    return oof, test_pred


print("uji vote:", knn_vote(np.array([[0.9, 0.1, 0.8]]), np.array([0, 1, 1]), 2).round(2))

## gzip-kNN (jarak kompresi normal, NCD)
NCD(a, b) = (C(ab) − min(C(a), C(b))) / max(C(a), C(b)), C = panjang hasil `zlib`.
Biaya = baris × `NCD_POOL` kompresi; turunkan `NCD_POOL` kalau lambat.

In [ ]:
encoded = []
for value in all_text:
    encoded.append(value.encode("utf-8"))
sizes = np.zeros(len(encoded))
for number, value in enumerate(encoded):
    sizes[number] = len(zlib.compress(value))


def gzip_predict(idx_fit, idx_query):
    rng = np.random.default_rng(SEED)
    pool = rng.choice(idx_fit, size=min(NCD_POOL, len(idx_fit)), replace=False)
    joined = np.zeros((len(idx_query), len(pool)))
    for row, query in enumerate(idx_query):
        for column, other in enumerate(pool):
            joined[row, column] = len(zlib.compress(encoded[query] + b" " + encoded[other]))
    small = np.minimum.outer(sizes[idx_query], sizes[pool])
    large = np.maximum.outer(sizes[idx_query], sizes[pool])
    return knn_vote(1 - (joined - small) / large, y[pool], NCD_K)


print("uji gzip:", gzip_predict(folds[0][0][:50], folds[0][1][:3]).round(2))

## BM25-kNN + Rocchio
BM25: bobot dokumen `idf × tf × (k1 + 1) / (tf + k1 × (1 − b + b × panjang / rata2))`, kueri
biner. Skor = perkalian matriks (cepat, tanpa paket `rank_bm25`).

In [ ]:
def bm25_weights(term_counts, k1=1.5, b_value=0.75):
    weights = term_counts.tocsr().astype(float)
    lengths = np.asarray(weights.sum(axis=1)).ravel()
    doc_freq = np.bincount(weights.indices, minlength=weights.shape[1])
    idf = np.log(1 + (weights.shape[0] - doc_freq + 0.5) / (doc_freq + 0.5))
    row_norm = k1 * (1 - b_value + b_value * lengths / lengths.mean())
    rows = np.repeat(np.arange(weights.shape[0]), np.diff(weights.indptr))
    term_freq = weights.data
    weights.data = idf[weights.indices] * term_freq * (k1 + 1) / (term_freq + row_norm[rows])
    return weights


bm25_docs = bm25_weights(counts)
query_binary = (counts > 0).astype(float)
print("BM25 siap:", bm25_docs.shape)

In [ ]:
def bm25_predict(idx_fit, idx_query):
    similarity = (query_binary[idx_query] @ bm25_docs[idx_fit].T).toarray()
    return knn_vote(similarity, y[idx_fit], BM25_K)


def rocchio_predict(idx_fit, idx_query):
    centroids = np.zeros((len(CLASSES), tfidf.shape[1]))
    for number in range(len(CLASSES)):
        members = idx_fit[y[idx_fit] == number]
        centroids[number] = np.asarray(tfidf[members].mean(axis=0)).ravel()
    similarity = tfidf[idx_query] @ normalize(centroids).T
    return softmax(np.asarray(similarity) * ROCCHIO_TEMP)


print("uji bm25:", bm25_predict(folds[0][0], folds[0][1][:2]).round(2))

## Complement NB, passive-aggressive, kernel RFF, patokan SGD
`PassiveAggressiveClassifier` usang di sklearn 1.8+: setaranya `SGDClassifier(loss="hinge",
penalty=None, learning_rate="pa1", eta0=1.0)`.

In [ ]:
kernel_rows = RBFSampler(gamma=RFF_GAMMA, n_components=2048, random_state=SEED).fit_transform(svd)


def cnb_predict(idx_fit, idx_query):
    fitted = ComplementNB(alpha=0.3).fit(counts[idx_fit], y[idx_fit])
    return fitted.predict_proba(counts[idx_query])


def pa_predict(idx_fit, idx_query):
    fitted = SGDClassifier(loss="hinge", penalty=None, learning_rate="pa1", eta0=1.0,
                           random_state=SEED).fit(tfidf[idx_fit], y[idx_fit])
    scores = fitted.decision_function(tfidf[idx_query])
    if scores.ndim == 1:
        scores = np.c_[-scores, scores]
    return softmax(scores * 3)


def rff_predict(idx_fit, idx_query):
    fitted = SGDClassifier(loss="log_loss", alpha=1e-4, random_state=SEED)
    fitted.fit(kernel_rows[idx_fit], y[idx_fit])
    return fitted.predict_proba(kernel_rows[idx_query])


def sgd_predict(idx_fit, idx_query):
    fitted = SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)
    fitted.fit(tfidf[idx_fit], y[idx_fit])
    return fitted.predict_proba(tfidf[idx_query])


print("kernel RFF:", kernel_rows.shape)

## Jalankan semua (lipatan sama)

In [ ]:
MEMBERS = {"sgd tfidf (patokan)": (sgd_predict, None), "gzip-kNN": (gzip_predict, "alt_gzip"),
           "BM25-kNN": (bm25_predict, "alt_bm25"), "Rocchio": (rocchio_predict, "alt_rocchio"),
           "complement NB": (cnb_predict, "alt_cnb"), "passive-aggressive": (pa_predict, "alt_pa"),
           "kernel RFF": (rff_predict, "alt_rff")}
RESULTS = {}
SECONDS = {}
for name, (predict, _) in MEMBERS.items():
    start = time.time()
    RESULTS[name] = run_member(predict)
    SECONDS[name] = round(time.time() - start)
    print(f"{name:20s} | OOF {metric(y, RESULTS[name][0]):.4f} | detik {SECONDS[name]}")

## Keragaman: berguna untuk ensemble?
- **+ patokan**: skor rata-rata model ini dengan SGD (naik = layak masuk ensemble).
- **menyelamatkan**: bagian baris yang SGD salah tapi model ini benar.
- **korelasi**: korelasi peluang dengan SGD (makin rendah makin beragam).

In [ ]:
base_oof = RESULTS["sgd tfidf (patokan)"][0]
base_wrong = base_oof.argmax(axis=1) != y
rows = []
for name in RESULTS:
    oof = RESULTS[name][0]
    rescued = (oof.argmax(axis=1) == y) & base_wrong
    rows.append({"model": name, "OOF": round(metric(y, oof), 4),
                 "+ patokan": round(metric(y, (oof + base_oof) / 2), 4),
                 "menyelamatkan": round(rescued.sum() / base_wrong.sum(), 3),
                 "korelasi": round(np.corrcoef(oof.ravel(), base_oof.ravel())[0, 1], 3),
                 "detik": SECONDS[name]})
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUTPUT_FOLDER}/banding_model.csv", index=False)
print(summary.to_string(index=False))

## Simpan OOF (n × K, kolom = `np.unique(train[TARGET])`) + catat percobaan
Nama file `oof_alt_*.npy`; masukkan ke `MODELS` di `ensemble_oof` atau
`bobot_ensemble_metaheuristik`.

In [ ]:
log_path = f"{OOF_FOLDER}/percobaan.csv"
for name, (_, slug) in MEMBERS.items():
    if slug is not None:
        oof, test_pred = RESULTS[name]
        np.save(f"{OOF_FOLDER}/oof_{slug}.npy", oof.astype("float32"))
        np.save(f"{OOF_FOLDER}/test_{slug}.npy", test_pred.astype("float32"))
        log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                                 "notebook": "pengklasifikasi_alternatif", "model": slug,
                                 "metrik": "f1_macro", "skor_cv": round(metric(y, oof), 4),
                                 "detik": SECONDS[name], "catatan": name}])
        log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("OOF ditulis ke", OOF_FOLDER)

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].astype(str).values, index=key[ID_COL].values)
truth_test = np.searchsorted(CLASSES, test[ID_COL].map(key_by_id).to_numpy())
base_test = RESULTS["sgd tfidf (patokan)"][1]
for name in RESULTS:
    test_pred = RESULTS[name][1]
    print(f"{name:20s} | test {metric(truth_test, test_pred):.4f}",
          f"| + patokan {metric(truth_test, (test_pred + base_test) / 2):.4f}")